In [ ]:
# This installs all four Azure packages you need for the entire course

%pip install python-dotenv
%pip install azure-ai-inference
%pip install azure-identity  
%pip install azure-ai-projects>= 2.0.0
%pip install openai

In [1]:
import os
from dotenv import load_dotenv

# load_dotenv() reads your .env file and loads the values into Python
# It looks for .env in the same folder as this notebook
load_dotenv()

# Read the values — they now come from .env, not from this code
endpoint   = os.environ["AZURE_OPENAI_ENDPOINT"]
key        = os.environ["AZURE_OPENAI_KEY"]
deployment = os.environ["AZURE_DEPLOYMENT_NAME"]

# Confirm they loaded — never print the actual key
print("Endpoint :", endpoint)
print("Key set? :", "Yes ✅" if key else "No ❌")
print("Model    :", deployment)

Endpoint : https://abhishekbhalotia333-344-resource.openai.azure.com/
Key set? : Yes ✅
Model    : gpt-4.1


In [2]:
import os
from openai import AzureOpenAI

In [3]:
# ── Read values from environment variables (set in Cell 2) ────────────────
endpoint    = os.environ["AZURE_OPENAI_ENDPOINT"]
key         = os.environ["AZURE_OPENAI_KEY"]
deployment  = os.environ["AZURE_DEPLOYMENT_NAME"]

In [4]:
# ── Create the client ─────────────────────────────────────────────────────
# AzureOpenAI is the Azure-specific version of the OpenAI client
# api_version tells Azure which version of its API to use
client = AzureOpenAI(
    azure_endpoint = endpoint,
    api_key        = key,
    api_version    = "2024-12-01-preview"
)

In [5]:
# ── Send your first message to gpt-4.1 ───────────────────────────────────
print("Calling gpt-4.1 on Azure AI Foundry...\n")

response = client.chat.completions.create(
    model    = deployment,   # your deployment name
    messages = [
        {"role": "system", "content": "You are a helpful AI tutor. Explain clearly and simply."},
        {"role": "user",   "content": "Explain what Azure AI Foundry is in one sentence."}
    ]
)

Calling gpt-4.1 on Azure AI Foundry...



In [6]:
# ── Read and print the answer ─────────────────────────────────────────────
answer = response.choices[0].message.content
tokens = response.usage.total_tokens

print("── gpt-4.1 says: ──────────────────────────────────")
print(answer)
print(f"\n── Tokens used: {tokens} ──────────────────────────────")

── gpt-4.1 says: ──────────────────────────────────
Azure AI Foundry is a Microsoft platform that helps organizations build, customize, deploy, and manage AI models efficiently using a unified set of AI tools and cloud infrastructure.

── Tokens used: 68 ──────────────────────────────


Phase 1 - Module 3 - "Develop a generative AI chat app with Microsoft Foundry"

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
%pip install "azure-ai-projects>=2.0.0" azure-identity python-dotenv

In [4]:
# ── Cell 2: Imports and credentials ─────────────────────────────────────────
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient ##Using Foundry-specific client SDK

load_dotenv()

True

In [5]:
# ── Cell 3: Create the project client ────────────────────────────────────────
project_client = AIProjectClient(
    endpoint   = os.environ["PROJECT_ENDPOINT"],
    credential = DefaultAzureCredential()
)

In [6]:
# ── Cell 4: Get an OpenAI client FROM the project client ─────────────────────
openai_client = project_client.get_openai_client()

In [7]:
# ── Cell 5: Call the model ───────────────────────────────────────────────────
response = openai_client.responses.create(
    model = os.environ["MODEL_DEPLOYMENT_NAME"],   # your deployment name
    input = "Explain Azure AI Foundry in one sentence."
)

print(response.output_text)

Azure AI Foundry is a unified platform from Microsoft that streamlines the building, customization, and deployment of generative AI solutions at scale using a robust set of tools, models, and infrastructure on Azure.


LP1 Module 4 — Develop Generative AI Apps That Use Tools

In [1]:
## Phase 1 - Module 4 — Tools: code_interpreter

In [9]:
# — Cell 7: Tool calling with code_interpreter ——————————————————

# No new imports needed — project_client and openai_client 
# are already created in Cells 3 and 4 above.
# Just make sure you ran those cells first.

response = openai_client.responses.create(
    model = os.environ["MODEL_DEPLOYMENT_NAME"],
    
    # This is the only new thing vs Module 3
    # We're telling the model: you have code_interpreter available
    # "type": "auto" tells Azure: "spin up a fresh sandboxed container for this conversation automatically." 
    # You don't manage it — Azure creates it, runs the code, and cleans it up.
    tools = [{"type": "code_interpreter", "container": {"type": "auto"}}],
    
    # Ask something that genuinely needs calculation
    # A good model will use the tool rather than guess
    input = (
        "I invest ₹1,00,000 at 8.5% compound interest annually. "
        "Show me the value at the end of each year for 5 years, "
        "and tell me the total interest earned."
    )
)

print(response.output_text)

Let's calculate the Compound Interest for 5 years:

**Formula:**  
A = P × (1 + r/100)ⁿ  
- A = Amount after n years  
- P = Principal = ₹1,00,000  
- r = Rate of Interest = 8.5%  
- n = number of years

Let's find out the amount at the end of each year and the total interest earned.Here are the details:

### Value at the End of Each Year:
1. **End of Year 1:** ₹1,08,500.00
2. **End of Year 2:** ₹1,17,722.50
3. **End of Year 3:** ₹1,27,728.91
4. **End of Year 4:** ₹1,38,585.87
5. **End of Year 5:** ₹1,50,365.67

### Total Interest Earned in 5 Years:
- **₹50,365.67**

If you'd like this in a table or chart, let me know!


In [10]:
# — Cell 8: Inspect what the model actually did ——————————————————

# response.output is a list of everything that happened
# Let's look at each item
for item in response.output:
    print(f"Type: {item.type}")
    
    # If the model called code_interpreter, show the code it wrote
    if item.type == "code_interpreter_call":
        print(f"Code the model wrote and executed:")
        print("─" * 40)
        print(item.code)
        print("─" * 40)
    
    # The final text response
    if item.type == "message":
        print(f"Final answer given to user")

Type: message
Final answer given to user
Type: code_interpreter_call
Code the model wrote and executed:
────────────────────────────────────────
# Variables
P = 100000  # Principal
r = 8.5     # Rate of interest
years = 5   # Number of years

# List to store year-end amounts
amounts = []

# Calculating the amount at the end of each year
for n in range(1, years+1):
    A = P * (1 + r/100) ** n
    amounts.append(round(A, 2))

# Total interest earned after 5 years
total_interest = amounts[-1] - P

amounts, total_interest
────────────────────────────────────────
Type: message
Final answer given to user
